<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 7 — Python Toolkit</div>
</div>

#### Random numbers, log-probabilities, percentiles, autocorrelation, and the emcee and ArviZ libraries: the Python behind today's sampler.
***

## Why this notebook

Six pieces of Python for today's lab. Twenty minutes; skim what you know. Section 7 has optional practice with numbers to check against. If arrays, slicing, loops or functions themselves feel shaky, start with the matching section of the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) first.

1. random numbers with `np.random.default_rng`: reproducible, and the two kinds the sampler needs;
2. log-probabilities: why products of probabilities underflow, and sums of logs don't;
3. `-np.inf`: how to say "impossible" in log-probability;
4. percentiles: medians and credible intervals from samples;
5. the autocorrelation of a sequence, and how many independent values it holds;
6. library samplers: `emcee`, checked with `ArviZ` (Lecture 9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## <span style="color:#B5121B">1. Random numbers</span>

Make one generator with a **seed** and draw everything from it: the same seed gives the same numbers every run, so your results are reproducible (and checkable against the numbers in the lab). The sampler needs two kinds: Gaussian numbers (`standard_normal`, mean 0 and standard deviation 1, for the proposed steps) and uniform numbers between 0 and 1 (`uniform`, for the accept/reject decision).

In [ ]:
rng = np.random.default_rng(42)
print("three Gaussian numbers:", rng.standard_normal(3).round(3))
print("one uniform number:    ", round(rng.uniform(), 3))
steps = np.array([3.0, 0.004, 0.6])
print("a step with a different width in each direction:", (steps * rng.standard_normal(3)).round(4))
rng = np.random.default_rng(42)
print("same seed, same numbers:", rng.standard_normal(3).round(3))

## <span style="color:#B5121B">2. Work with log-probabilities</span>

A likelihood is a product of one probability per data point. With many points the product becomes too small for a computer to store (it **underflows** to 0), and then every ratio is 0/0. The logarithm turns the product into a sum, which is always safe. The ratio of two probabilities becomes a difference of logs: $p'/p = e^{\ln p' - \ln p}$.

In [ ]:
probs = np.full(400, 0.1)                # 400 data points, each with probability 0.1
print("product:", np.prod(probs))           # underflows: the true value is 1e-400
print("sum of logs:", np.log(probs).sum())  # -921.0: no problem
# comparing two models: ln p1 - ln p2 is fine; the ratio of the raw products is 0/0
print("log of the ratio:", np.log(probs).sum() - np.log(np.full(400, 0.11)).sum())

## <span style="color:#B5121B">3. Saying "impossible": `-np.inf`</span>

A prior that is zero outside some limits has log $-\infty$. NumPy handles it: any finite number is greater than `-np.inf`, and `np.exp(-np.inf)` is 0. Return it **before** computing the likelihood, so the model is never evaluated where it might fail (a negative σ, for example).

In [ ]:
def log_prior(sigma):
    if not 0 < sigma < 50:
        return -np.inf
    return 0.0                                   # flat inside the limits (up to a constant)
print(log_prior(5), log_prior(-1), np.exp(log_prior(-1)))
print("is a log-probability of -1e6 better than impossible?", -1e6 > -np.inf)

In [ ]:
# Your turn: why does the sampler in the lab never accept a step outside the prior? (Hint: what is lp_prop - lp?)

## <span style="color:#B5121B">4. Percentiles: credible intervals from samples</span>

Once you have samples from a distribution, summarise it with **percentiles**: the median is the 50th, and the central 68% interval runs from the 16th to the 84th (the analogue of $\pm 1\sigma$). `axis=0` gives one value per column, so a chain with one column per parameter gives every parameter at once. For a Gaussian, these match the mean and standard deviation; for a skewed distribution they don't, and the percentiles are the honest summary.

In [ ]:
rng = np.random.default_rng(0)
samples = np.column_stack([rng.normal(10, 2, 20000), rng.lognormal(0, 0.5, 20000)])
lo, med, hi = np.percentile(samples, [16, 50, 84], axis=0)
for k, name in enumerate(["Gaussian", "skewed"]):
    print(f"{name:9s} median {med[k]:.2f}, 68% interval {lo[k]:.2f} to {hi[k]:.2f};  mean ± sd {samples[:, k].mean():.2f} ± {samples[:, k].std():.2f}")

## <span style="color:#B5121B">5. Autocorrelation and the effective number of samples</span>

The **autocorrelation** at lag $\ell$ is the correlation between a sequence and itself shifted by $\ell$ steps: $\rho_0 = 1$, and for independent values $\rho_\ell \approx 0$ for $\ell \geq 1$. A random walk that moves a little at a time stays correlated for many steps. The effective number of independent values is $N_\mathrm{eff} = N / (1 + 2\sum_{\ell\ge 1}\rho_\ell)$. The FFT computes all lags at once.

In [ ]:
def autocorr(x):
    x = x - x.mean()
    f = np.fft.rfft(x, 2 * len(x))               # padding to 2N avoids wrapping round
    ac = np.fft.irfft(f * np.conj(f))[:len(x)]
    return ac / ac[0]

rng = np.random.default_rng(1)
white = rng.standard_normal(5000)                # independent values
sticky = np.zeros(5000)                          # each value is 0.95 of the last, plus noise
for i in range(1, 5000):
    sticky[i] = 0.95 * sticky[i - 1] + rng.standard_normal()
for x, name in [(white, "independent"), (sticky, "sticky")]:
    rho = autocorr(x)
    k = np.argmax(rho < 0.05)
    print(f"{name:12s} rho at lag 1: {rho[1]:.2f};  N_eff = {len(x) / (1 + 2 * rho[1:k].sum()):.0f} of {len(x)}")
plt.figure(figsize=(6, 3)); plt.plot(autocorr(white)[:100], label="independent"); plt.plot(autocorr(sticky)[:100], label="sticky")
plt.axhline(0.05, color="k", ls=":"); plt.xlabel("lag"); plt.ylabel("autocorrelation"); plt.legend(); plt.show()

## <span style="color:#B5121B">6. Library samplers: emcee, checked with ArviZ</span>

Writing your own sampler shows you how it works; for real problems use a tested library (Lecture 9). **emcee** runs many *walkers* at once, each proposing moves using the positions of the others, so it adapts to correlated parameters without step widths to tune. **ArviZ** reads the result and reports Lecture 9's checks: `r_hat` ($\hat R$, which should be 1.01 or less) and `ess_bulk` (the effective number of samples, $N_\mathrm{eff}$). Neither comes with Python, so the first cell installs them if needed.

The example: fit a straight line $y = a + bx$ to 20 points with uncertainty 1, flat priors.

In [ ]:
import importlib, subprocess, sys
for pkg in ("emcee", "arviz"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import emcee
import arviz as az

rng = np.random.default_rng(6)
x = np.linspace(0, 10, 20)
y = 2.0 + 0.5 * x + rng.normal(0, 1, 20)          # true intercept 2, slope 0.5, uncertainty 1

def log_post(theta):
    a, b = theta
    if not (-10 < a < 10 and -5 < b < 5):          # flat priors
        return -np.inf
    return -0.5 * np.sum((y - a - b * x)**2)       # sigma = 1

nwalkers = 16
start = 0.01 * rng.standard_normal((nwalkers, 2))  # all walkers start in a small ball near (0, 0)
sampler = emcee.EnsembleSampler(nwalkers, 2, log_post)
sampler.random_state = np.random.RandomState(6).get_state()   # emcee keeps its own random generator: seed it too
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain()                        # shape (steps, walkers, parameters)
print(chain.shape)
plt.figure(figsize=(7, 3)); plt.plot(chain[:800, :, 1], lw=0.5)
plt.xlabel("step"); plt.ylabel("slope b"); plt.title("16 walkers: the start is forgotten within a few hundred steps"); plt.show()

Discard the start (here the first 500 steps) and pool all walkers: `get_chain(discard=500, flat=True)` gives one row per sample, ready for `np.percentile` (Section 4). Then hand the sampler to ArviZ, which treats each walker as a chain. `kind="diagnostics"` prints only the checks: the full `az.summary` also gives intervals, but their width depends on the ArviZ version (94% in older versions, 89% in newer) and newer versions round them heavily, so take intervals from the samples yourself.

In [ ]:
flat = sampler.get_chain(discard=500, flat=True)
lo, med, hi = np.percentile(flat, [16, 50, 84], axis=0)
b_ls, a_ls = np.polyfit(x, y, 1)
for k, (name, ls) in enumerate([("intercept a", a_ls), ("slope b", b_ls)]):
    print(f"{name:11s} median {med[k]:.3f}, 68% interval {lo[k]:.3f} to {hi[k]:.3f};  least squares {ls:.3f}")
print("acceptance fraction:", sampler.acceptance_fraction.mean().round(2))

idata = az.from_emcee(sampler, var_names=["a", "b"])
print(az.summary(idata.sel(draw=slice(500, None)), kind="diagnostics"))

In [ ]:
# Your turn: print the diagnostics for only the first 300 steps (slice(0, 300)). What happens to r_hat and ess_bulk, and why?
# Then: why does it hardly matter here whether you discard the first 500 of 3000 steps?

## <span style="color:#FD9029">7. Optional practice</span>

Four short problems. Check your numbers against the **Check** lines.

**P1.** Use `rng = np.random.default_rng(7)` and draw 100 000 uniform numbers. What fraction is below 0.25? Below $e^{-1}$? (This is the chance that the sampler accepts a step that lowers the log-posterior by 1.)

In [ ]:
# P1

**Check:** about 0.250 and 0.368 ($e^{-1}$). `np.log(u) < -1` gives the same fraction, which is why the sampler can compare logs.

**P2.** The Gaussian log-likelihood for 12 points with residuals `r` and uncertainty σ is $-\tfrac12\sum (r_i/\sigma)^2 - 12\ln\sigma$ (up to a constant). For residuals `r = np.random.default_rng(3).normal(0, 4, 12)`, evaluate it for σ from 1 to 20 and find where it peaks. What happens without the $-12\ln\sigma$ term?

In [ ]:
# P2

**Check:** the peak is at the root-mean-square of the residuals (about 6.1 for these numbers: a small sample can scatter well away from the true 4). Without the $\ln\sigma$ term the log-likelihood just keeps rising, to the largest σ allowed (20 here).

**P3.** Draw 50 000 samples from a Gaussian with mean 5 and standard deviation 2, and give the median and 68% interval. Then square each sample and do the same. Is the median of the squares the square of the median? Is the interval symmetric?

In [ ]:
# P3

**Check:** $x$: 5.00 (−2.0, +2.0). $x^2$: about 25.0 (−16, +24): the median transforms with the function (5² = 25) but the interval becomes lopsided. Samples carry this through automatically; error propagation by derivatives would not.

**P4.** For the 'sticky' sequence in Section 5, change 0.95 to 0.5 and to 0.99. How does $N_\mathrm{eff}$ change? For this kind of sequence, theory says $N_\mathrm{eff} \approx N(1-a)/(1+a)$ for coefficient $a$.

In [ ]:
# P4

**Check:** roughly 1700, 110 and 30 (theory 1667, 128, 25). The estimates are noisy for very sticky sequences: with $a = 0.99$ there are only a few independent stretches to measure.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>